# Laboratorio Spark: salarios de personas asalariadas en Guatemala (ENEIC)

## Parte 1. Carga, preparación y auditoría de los datos

**Herramientas:** Python, Spark 3.5 (DataFrames y `pyspark.ml`).
**Fuente:** Encuesta Nacional de Empleo e Ingresos Continua (ENEIC), bases de *Personas*, INE Guatemala.
**Uso de los datos:** los cuatro trimestres de 2025 forman el conjunto de desarrollo (*train*) y el primer trimestre de 2026 es la prueba final (*test*).

En esta parte se hace lo siguiente:

1. Configurar Spark y las rutas.
2. Leer cada archivo por separado, seleccionar las columnas requeridas, homologar tipos y guardar en Parquet.
3. Unir los cuatro archivos de 2025 con `unionByName` y auditar la unión (esquema, conteos, faltantes, claves).
4. Aplicar los filtros de la población analítica, siempre en el mismo orden, contando las exclusiones de cada paso.
5. Validar las variables categóricas y construir la antigüedad en años.
6. Guardar el conjunto preparado de 2025 y el de 2026 por separado en Parquet.

Todas las estadísticas y conteos se calculan con Spark sobre el conjunto completo. A pandas solo se pasan tablas pequeñas ya agregadas.

> **Cómo ejecutarlo:** edite únicamente la sección 0 (rutas y nombres de archivo). El resto corre de principio a fin sin depender de variables creadas antes.

## 0. Configuración y sesión de Spark

In [1]:
# Si se usa Colab u otro entorno sin Spark, descomente la siguiente línea:
# !pip install -q pyspark==3.5.1 openpyxl

import json
import math
import re
from functools import reduce
from pathlib import Path

import pandas as pd
from IPython.display import display

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql import SparkSession; s = SparkSession.builder.master("local[*]").getOrCreate(); print(s.version) #REVISAR PYSPARK NO FUNCIONABA ESTA BASURA
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

3.5.1


In [2]:

DIR_DATOS  = Path("datos").resolve()    
DIR_SALIDA = Path("salidas").resolve()  
HOJA       = 0                            

ARCHIVOS = {
    "2025T1": "Personas_ENEIC_T1_2025.xlsx",
    "2025T2": "Personas-ENEIC-T2-2025.xlsx",
    "2025T3": "Base-de-datos-Personas-ENEIC-III-2025.xlsx",
    "2025T4": "Base-de-datos-Personas-ENEIC-IV-2025.xlsx",
    "2026T1": "Base-de-datos-Personas-ENEIC-I-2026.xlsx",
}
PERIODOS_TRAIN = ["2025T1", "2025T2", "2025T3", "2025T4"]
PERIODO_TEST   = "2026T1"


FORZAR_LECTURA = False


ESPERADOS = {
    "2025T1": (51588, 270),
    "2025T2": (51167, 270),
    "2025T3": (51583, 270),
    "2025T4": (49338, 302),
    "2026T1": (49843, 270),
}

ETIQUETAS_NIVEL_EDUCATIVO = {   
    "0": "Ninguno", "1": "Preprimaria", "2": "Primaria", "3": "Básico",
    "4": "Diversificado", "5": "Superior", "6": "Maestría", "7": "Doctorado",
}
ETIQUETAS_DOMINIO = {           
    "1": "Urbano Metropolitano", "2": "Resto Urbano", "3": "Rural Nacional",
}

CODIGOS_VALIDOS = {
    "nivel_educativo": list(ETIQUETAS_NIVEL_EDUCATIVO),
    "dominio": list(ETIQUETAS_DOMINIO),
}


CATEGORIAS_ASALARIADO = {
    "1": "Empleado de gobierno",
    "2": "Empleado de empresa privada",
    "3": "Empleado jornalero o peón",
    "4": "Servicio doméstico",
}

In [3]:
spark = (
    SparkSession.builder
    .appName("ENEIC_salarios_parte1")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

(DIR_SALIDA / "raw").mkdir(parents=True, exist_ok=True)
(DIR_SALIDA / "preparado").mkdir(parents=True, exist_ok=True)

Spark 3.5.1


## 1. Carga de los archivos, homologación de tipos y Parquet

**Estrategia:**

- Spark no lee Excel de forma nativa. Cada archivo se lee **individualmente** con pandas/openpyxl, conservando solo las 14 columnas necesarias (así no se carga en memoria las 270 o 302 columnas originales).
- Los nombres de columna se normalizan (mayúsculas y sin espacios) y se buscan **por nombre**, nunca por posición.
- Los **códigos** (`OCUPADOS`, `P05C16`, `P03A03A`, `DOMINIO`, `NUM_HOGAR`, `NUM_PERSONA`) se convierten a una representación de texto consistente: `1`, `1.0` y `"01"` pasan a ser `"1"`. Un mismo código puede llegar como número o como texto. El código `0` se conserva como `"0"`; no se trata como faltante.
- Las variables **numéricas** se convierten a `double`. Lo que no puede interpretarse como número queda como nulo (así se cuenta como faltante y no se evalúa como válido).
- Cada archivo se convierte a un DataFrame de Spark con **esquema explícito** y se guarda en Parquet. Se agregan `archivo_origen`, `periodo_archivo`, `anio_archivo` y `trimestre_calendario`, tomados del **archivo de procedencia**. La columna original `TRIMESTRE` se conserva tal cual y no se usa para el calendario.

| Variable original | Nombre analítico | Uso |
|---|---|---|
| P05D01 | salario_mensual | Variable objetivo |
| P02A03 | edad | Predictor numérico |
| P05C07A / P05C07B | antiguedad_anios / antiguedad_meses | Construcción de antigüedad |
| P05H01A | horas_semanales | Predictor y clustering |
| P03A03A | nivel_educativo | Predictor categórico |
| P05C16 | categoria_ocupacional | Filtro y predictor categórico |
| DOMINIO | dominio | Predictor categórico |
| OCUPADOS | ocupado | Filtro |
| NUM_HOGAR, NUM_PERSONA | (se conservan) | Auditoría de registros |
| FACTOR | (se conserva) | Documentación del diseño muestral |
| ANIO, TRIMESTRE | (se conservan) | Auditoría de la fuente |

In [4]:
COLUMNAS_ORIGINALES = {
    "TRIMESTRE", "ANIO", "NUM_HOGAR", "NUM_PERSONA", "FACTOR", "OCUPADOS",
    "P02A03", "P05C07A", "P05C07B", "P05H01A", "P03A03A", "P05C16", "DOMINIO", "P05D01",
}

# Esquema explícito de cada DataFrame de Spark (orden de columnas fijo)
SCHEMA = T.StructType([
    T.StructField("archivo_origen",          T.StringType()),
    T.StructField("periodo_archivo",         T.StringType()),
    T.StructField("anio_archivo",            T.IntegerType()),
    T.StructField("trimestre_calendario",    T.IntegerType()),
    T.StructField("ANIO",                    T.IntegerType()),   # original
    T.StructField("TRIMESTRE",               T.IntegerType()),   # original
    T.StructField("NUM_HOGAR",               T.StringType()),
    T.StructField("NUM_PERSONA",             T.StringType()),
    T.StructField("FACTOR",                  T.DoubleType()),
    T.StructField("ocupado",                 T.StringType()),    # OCUPADOS
    T.StructField("edad",                    T.DoubleType()),    # P02A03
    T.StructField("antiguedad_anios",        T.DoubleType()),    # P05C07A
    T.StructField("antiguedad_meses",        T.DoubleType()),    # P05C07B
    T.StructField("horas_semanales",         T.DoubleType()),    # P05H01A
    T.StructField("nivel_educativo",         T.StringType()),    # P03A03A
    T.StructField("categoria_ocupacional",   T.StringType()),    # P05C16
    T.StructField("dominio",                 T.StringType()),    # DOMINIO
    T.StructField("salario_mensual",         T.DoubleType()),    # P05D01
])
COLUMNAS_SELECCIONADAS = [f.name for f in SCHEMA.fields]
VARIABLES_ANALITICAS = [c for c in COLUMNAS_SELECCIONADAS
                        if c not in ("archivo_origen", "periodo_archivo", "anio_archivo", "trimestre_calendario")]

In [5]:

def parse_periodo(periodo):
    # "2025T1" -> (2025, 1)
    m = re.fullmatch(r"(\d{4})T([1-4])", periodo)
    if not m:
        raise ValueError(f"Periodo con formato inesperado: {periodo!r} (se espera, por ejemplo, '2025T1')")
    return int(m.group(1)), int(m.group(2))


def normalizar_codigo(x):

    if x is None or x is pd.NA:
        return None
    if isinstance(x, float) and math.isnan(x):
        return None
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null", "<na>"}:
        return None
    try:
        v = float(s)
        if math.isfinite(v) and v == int(v):
            return str(int(v))
    except ValueError:
        pass
    return s.upper()


def leer_seleccion(ruta):

    ruta = Path(ruta)
    es_csv = ruta.suffix.lower() == ".csv"
    if es_csv:
        encabezado = pd.read_csv(ruta, nrows=0)
    else:
        encabezado = pd.read_excel(ruta, sheet_name=HOJA, nrows=0)
    columnas_up = [str(c).strip().upper() for c in encabezado.columns]

    ausentes = sorted(COLUMNAS_ORIGINALES - set(columnas_up))
    if ausentes:
        raise ValueError(f"{ruta.name}: faltan columnas requeridas: {ausentes}")

    filtro = lambda c: str(c).strip().upper() in COLUMNAS_ORIGINALES
    if es_csv:
        pdf = pd.read_csv(ruta, usecols=filtro, dtype=object)
    else:
        pdf = pd.read_excel(ruta, sheet_name=HOJA, usecols=filtro, dtype=object)
    pdf.columns = [str(c).strip().upper() for c in pdf.columns]
    if pdf.columns.duplicated().any():
        raise ValueError(f"{ruta.name}: nombres de columna repetidos tras normalizar: "
                         f"{list(pdf.columns[pdf.columns.duplicated()])}")

    num = lambda s: pd.to_numeric(s, errors="coerce")
    out = pd.DataFrame({
        "ANIO":                  num(pdf["ANIO"]),
        "TRIMESTRE":             num(pdf["TRIMESTRE"]),
        "NUM_HOGAR":             pdf["NUM_HOGAR"].map(normalizar_codigo),
        "NUM_PERSONA":           pdf["NUM_PERSONA"].map(normalizar_codigo),
        "FACTOR":                num(pdf["FACTOR"]),
        "ocupado":               pdf["OCUPADOS"].map(normalizar_codigo),
        "edad":                  num(pdf["P02A03"]),
        "antiguedad_anios":      num(pdf["P05C07A"]),
        "antiguedad_meses":      num(pdf["P05C07B"]),
        "horas_semanales":       num(pdf["P05H01A"]),
        "nivel_educativo":       pdf["P03A03A"].map(normalizar_codigo),
        "categoria_ocupacional": pdf["P05C16"].map(normalizar_codigo),
        "dominio":               pdf["DOMINIO"].map(normalizar_codigo),
        "salario_mensual":       num(pdf["P05D01"]),
    })
    meta = {"archivo": ruta.name, "n_registros": int(len(pdf)),
            "n_columnas": len(columnas_up), "columnas": columnas_up}
    return out, meta


def _a_python(v, tipo):

    if v is None or v is pd.NA:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    if isinstance(tipo, T.IntegerType):
        return int(v)
    if isinstance(tipo, T.DoubleType):
        return float(v)
    return str(v)


def a_filas(pdf):
    tipos = [f.dataType for f in SCHEMA.fields]
    return [tuple(_a_python(v, t) for v, t in zip(fila, tipos))
            for fila in pdf[COLUMNAS_SELECCIONADAS].itertuples(index=False, name=None)]


def cargar_archivo(periodo):

    ruta_pq   = DIR_SALIDA / "raw" / f"{periodo}.parquet"
    ruta_meta = DIR_SALIDA / "raw" / f"{periodo}_meta.json"
    if ruta_pq.exists() and ruta_meta.exists() and not FORZAR_LECTURA:
        meta = json.loads(ruta_meta.read_text(encoding="utf-8"))
        return spark.read.parquet(str(ruta_pq)), meta

    archivo = DIR_DATOS / ARCHIVOS[periodo]
    pdf, meta = leer_seleccion(archivo)
    anio, trim = parse_periodo(periodo)
    pdf.insert(0, "archivo_origen", archivo.name)
    pdf.insert(1, "periodo_archivo", periodo)
    pdf.insert(2, "anio_archivo", anio)
    pdf.insert(3, "trimestre_calendario", trim)

    sdf = spark.createDataFrame(a_filas(pdf), schema=SCHEMA)
    sdf.write.mode("overwrite").parquet(str(ruta_pq))
    ruta_meta.write_text(json.dumps(meta), encoding="utf-8")
    del pdf
    return spark.read.parquet(str(ruta_pq)), meta

In [6]:
dfs, meta_archivos = {}, {}
for periodo in ARCHIVOS:
    dfs[periodo], meta_archivos[periodo] = cargar_archivo(periodo)
    m = meta_archivos[periodo]
    print(f"{periodo}: {m['archivo']:<40s} {m['n_registros']:>7,} registros | {m['n_columnas']} columnas originales")

2025T1: Personas_ENEIC_T1_2025.xlsx               51,588 registros | 270 columnas originales
2025T2: Personas-ENEIC-T2-2025.xlsx               51,167 registros | 270 columnas originales
2025T3: Base-de-datos-Personas-ENEIC-III-2025.xlsx  51,583 registros | 270 columnas originales
2025T4: Base-de-datos-Personas-ENEIC-IV-2025.xlsx  49,338 registros | 302 columnas originales
2026T1: Base-de-datos-Personas-ENEIC-I-2026.xlsx  49,843 registros | 270 columnas originales


### 1.1 Verificación contra los conteos del enunciado

In [7]:
verif = pd.DataFrame([{
    "periodo_archivo": p,
    "archivo_origen": meta_archivos[p]["archivo"],
    "registros_leidos": meta_archivos[p]["n_registros"],
    "registros_esperados": ESPERADOS[p][0],
    "columnas_leidas": meta_archivos[p]["n_columnas"],
    "columnas_esperadas": ESPERADOS[p][1],
    "coincide": (meta_archivos[p]["n_registros"], meta_archivos[p]["n_columnas"]) == ESPERADOS[p],
} for p in ARCHIVOS])
display(verif)
if not verif["coincide"].all():
    print("ATENCIÓN: hay archivos cuyos conteos no coinciden con el enunciado. Revise nombres de archivo y hoja (HOJA).")

,periodo_archivo,archivo_origen,registros_leidos,registros_esperados,columnas_leidas,columnas_esperadas,coincide
0,2025T1,Personas_ENEIC_T1_2025.xlsx,51588,51588,270,270,True
1,2025T2,Personas-ENEIC-T2-2025.xlsx,51167,51167,270,270,True
2,2025T3,Base-de-datos-Personas-ENEIC-III-2025.xlsx,51583,51583,270,270,True
3,2025T4,Base-de-datos-Personas-ENEIC-IV-2025.xlsx,49338,49338,302,302,True
4,2026T1,Base-de-datos-Personas-ENEIC-I-2026.xlsx,49843,49843,270,270,True


### 1.2 Posición de las variables en cada archivo original

La tabla muestra la posición (base 1) que ocupa cada variable requerida en el archivo original. Sirve para comprobar si una unión por posición sería segura.

In [8]:
pos = pd.DataFrame({
    p: {c: m["columnas"].index(c) + 1 for c in sorted(COLUMNAS_ORIGINALES)}
    for p, m in meta_archivos.items()
})
pos["misma_posicion_en_todos"] = pos.nunique(axis=1) == 1
display(pos)

base = meta_archivos["2025T1"]["columnas"]
for p in ARCHIVOS:
    cols = meta_archivos[p]["columnas"]
    extra   = [c for c in cols if c not in set(base)]
    ausente = [c for c in base if c not in set(cols)]
    print(f"{p}: {len(cols)} columnas | no están en 2025T1: {len(extra)} | de 2025T1 que no trae: {len(ausente)}")
    if extra and p == "2025T4":
        print("   Ejemplos de columnas adicionales:", extra[:10])

n_mov = int((~pos["misma_posicion_en_todos"]).sum())
print(f"\nVariables requeridas que cambian de posición entre archivos: {n_mov} de {len(pos)}")

,2025T1,2025T2,2025T3,2025T4,2026T1,misma_posicion_en_todos
ANIO,1,1,1,1,1,True
DOMINIO,3,3,3,3,3,True
FACTOR,5,5,5,5,5,True
NUM_HOGAR,4,4,4,4,4,True
NUM_PERSONA,7,7,7,7,7,True
OCUPADOS,266,266,266,298,266,False
P02A03,13,13,13,9,13,False
P03A03A,33,33,33,29,33,False
P05C07A,68,68,68,61,68,False
P05C07B,69,69,69,62,69,False


2025T1: 270 columnas | no están en 2025T1: 0 | de 2025T1 que no trae: 0
2025T2: 270 columnas | no están en 2025T1: 0 | de 2025T1 que no trae: 0
2025T3: 270 columnas | no están en 2025T1: 0 | de 2025T1 que no trae: 0
2025T4: 302 columnas | no están en 2025T1: 42 | de 2025T1 que no trae: 10
   Ejemplos de columnas adicionales: ['P07A01A', 'P07A01B', 'P07A01C', 'P07A02A', 'P07A02B', 'P07A02C', 'P07A03A', 'P07A03B', 'P07A03C', 'P07A04A']
2026T1: 270 columnas | no están en 2025T1: 0 | de 2025T1 que no trae: 0

Variables requeridas que cambian de posición entre archivos: 8 de 14


**¿Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos?**

El archivo de IV de 2025 trae 302 columnas y los otros cuatro traen 270. Un `union` de Spark empareja las columnas por su **posición**, no por su nombre; esto solo es correcto si la columna *k* significa lo mismo en todos los archivos.

- Con distinto número de columnas, `union` ni siquiera se puede ejecutar.
- Si se recortaran los archivos al mismo número de columnas, las variables que cambian de posición (véase la tabla anterior) quedarían **mezcladas**: en una misma columna quedarían, por ejemplo, edades de un archivo y códigos de otra pregunta de otro archivo. Si los tipos son compatibles, Spark no avisaría del error.

`unionByName` empareja por **nombre**, por lo que es seguro siempre que los nombres estén homologados (mayúsculas y espacios) y todos los DataFrames tengan las mismas columnas con los mismos tipos. Eso es lo que se hizo antes de unir: se seleccionaron por nombre las mismas 14 columnas de cada archivo y se les dio el mismo esquema.

## 2. Unión de los cuatro archivos de 2025 con `unionByName`

El conjunto de 2026 (prueba final) se mantiene **separado** y recibe exactamente el mismo tratamiento, pero no se usa para tomar decisiones de análisis.

In [9]:
df_2025 = reduce(lambda a, b: a.unionByName(b), [dfs[p] for p in PERIODOS_TRAIN]).cache()
df_2026 = dfs[PERIODO_TEST].cache()
df_todo = df_2025.unionByName(df_2026)

print(f"Registros 2025 (4 archivos): {df_2025.count():,}")
print(f"Registros 2026T1:            {df_2026.count():,}")

Registros 2025 (4 archivos): 203,676
Registros 2026T1:            49,843


### 2.1 Esquema y cinco registros de las columnas seleccionadas

In [10]:
df_2025.printSchema()
display(df_2025.limit(5).toPandas())

root
 |-- archivo_origen: string (nullable = true)
 |-- periodo_archivo: string (nullable = true)
 |-- anio_archivo: integer (nullable = true)
 |-- trimestre_calendario: integer (nullable = true)
 |-- ANIO: integer (nullable = true)
 |-- TRIMESTRE: integer (nullable = true)
 |-- NUM_HOGAR: string (nullable = true)
 |-- NUM_PERSONA: string (nullable = true)
 |-- FACTOR: double (nullable = true)
 |-- ocupado: string (nullable = true)
 |-- edad: double (nullable = true)
 |-- antiguedad_anios: double (nullable = true)
 |-- antiguedad_meses: double (nullable = true)
 |-- horas_semanales: double (nullable = true)
 |-- nivel_educativo: string (nullable = true)
 |-- categoria_ocupacional: string (nullable = true)
 |-- dominio: string (nullable = true)
 |-- salario_mensual: double (nullable = true)



,archivo_origen,periodo_archivo,anio_archivo,trimestre_calendario,ANIO,TRIMESTRE,NUM_HOGAR,NUM_PERSONA,FACTOR,ocupado,edad,antiguedad_anios,antiguedad_meses,horas_semanales,nivel_educativo,categoria_ocupacional,dominio,salario_mensual
0,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1,2025,2,3299,1,300.0,None,84.0,NaN,NaN,NaN,2,None,2,NaN
1,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1,2025,2,3299,2,300.0,None,78.0,NaN,NaN,NaN,2,None,2,NaN
2,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1,2025,2,15607,2,318.0,1,54.0,44.0,0.0,44.0,0,3,2,1680.0
3,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1,2025,2,15607,3,318.0,1,20.0,1.0,0.0,36.0,4,4,2,600.0
4,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1,2025,2,15607,1,318.0,1,60.0,22.0,0.0,20.0,0,5,2,NaN


### 2.2 Identificación del período
`periodo_archivo`, `anio_archivo` y `trimestre_calendario` provienen del **archivo de procedencia**. `TRIMESTRE` se conserva como vino. La tabla siguiente muestra los valores originales de `TRIMESTRE` por archivo.

In [11]:
display(
    df_todo.groupBy("periodo_archivo", "archivo_origen", "anio_archivo", "trimestre_calendario", "TRIMESTRE")
    .count().orderBy("periodo_archivo", "TRIMESTRE").toPandas()
)

,periodo_archivo,archivo_origen,anio_archivo,trimestre_calendario,TRIMESTRE,count
0,2025T1,Personas_ENEIC_T1_2025.xlsx,2025,1,2,51588
1,2025T2,Personas-ENEIC-T2-2025.xlsx,2025,2,2,175
2,2025T2,Personas-ENEIC-T2-2025.xlsx,2025,2,3,50992
3,2025T3,Base-de-datos-Personas-ENEIC-III-2025.xlsx,2025,3,4,51583
4,2025T4,Base-de-datos-Personas-ENEIC-IV-2025.xlsx,2025,4,5,49338
5,2026T1,Base-de-datos-Personas-ENEIC-I-2026.xlsx,2026,1,6,49843


Según el enunciado, `TRIMESTRE` vale 2, 3, 4, 5 y 6 para los cinco archivos, y el archivo II de 2025 contiene además 175 registros con valor 2. Por eso **no** se resta uno automáticamente a `TRIMESTRE`: esa operación no resolvería todos los registros observados. El período calendario se asigna por archivo, lo que identifica el corte publicado al que pertenece cada registro sin modificar la respuesta original.

## 3. Registros y faltantes antes de aplicar filtros

In [12]:
antes = (df_todo.groupBy("periodo_archivo", "archivo_origen").count()
         .orderBy("periodo_archivo").toPandas().rename(columns={"count": "registros_originales"}))
display(antes)

,periodo_archivo,archivo_origen,registros_originales
0,2025T1,Personas_ENEIC_T1_2025.xlsx,51588
1,2025T2,Personas-ENEIC-T2-2025.xlsx,51167
2,2025T3,Base-de-datos-Personas-ENEIC-III-2025.xlsx,51583
3,2025T4,Base-de-datos-Personas-ENEIC-IV-2025.xlsx,49338
4,2026T1,Base-de-datos-Personas-ENEIC-I-2026.xlsx,49843


In [13]:
def tabla_faltantes(df, columnas):
    # Cantidad y porcentaje de nulos (y NaN en columnas double) por variable, calculados en Spark
    total = df.count()
    exprs = []
    for c in columnas:
        cond = F.col(c).isNull()
        if isinstance(df.schema[c].dataType, T.DoubleType):
            cond = cond | F.isnan(F.col(c))
        exprs.append(F.sum(cond.cast("int")).alias(c))
    fila = df.agg(*exprs).collect()[0].asDict()
    return pd.DataFrame({
        "variable": columnas,
        "faltantes": [int(fila[c] or 0) for c in columnas],
        "pct_faltantes": [round(100 * (fila[c] or 0) / total, 2) if total else float("nan") for c in columnas],
    }).set_index("variable")


falt_2025 = tabla_faltantes(df_2025, VARIABLES_ANALITICAS)
falt_2026 = tabla_faltantes(df_2026, VARIABLES_ANALITICAS)
faltantes = falt_2025.join(falt_2026, lsuffix="_2025", rsuffix="_2026T1")
print("Faltantes por variable ANTES de aplicar filtros (todas las personas de cada archivo):")
display(faltantes)

Faltantes por variable ANTES de aplicar filtros (todas las personas de cada archivo):


,faltantes_2025,pct_faltantes_2025,faltantes_2026T1,pct_faltantes_2026T1
variable,,,,
ANIO,0,0.00,0,0.00
TRIMESTRE,0,0.00,0,0.00
NUM_HOGAR,0,0.00,0,0.00
NUM_PERSONA,0,0.00,0,0.00
FACTOR,0,0.00,0,0.00
ocupado,115454,56.69,28109,56.40
edad,0,0.00,0,0.00
antiguedad_anios,115454,56.69,28109,56.40
antiguedad_meses,115454,56.69,28109,56.40


La tabla anterior mezcla dos situaciones muy distintas. Para separarlas, la siguiente tabla mide los faltantes **solo entre las personas a quienes sí corresponden las preguntas laborales**: ocupadas y con categoría ocupacional de asalariado (códigos 1 a 4).

In [14]:
def es_asalariado_ocupado():
    return (F.col("ocupado") == "1") & F.col("categoria_ocupacional").isin(list(CATEGORIAS_ASALARIADO))

vars_laborales = ["edad", "antiguedad_anios", "antiguedad_meses", "horas_semanales",
                  "nivel_educativo", "dominio", "salario_mensual"]
sub_2025 = df_2025.filter(es_asalariado_ocupado())
sub_2026 = df_2026.filter(es_asalariado_ocupado())
print(f"Ocupados asalariados (antes de filtros de calidad): 2025 = {sub_2025.count():,} | 2026T1 = {sub_2026.count():,}")
falt_sub = tabla_faltantes(sub_2025, vars_laborales).join(
    tabla_faltantes(sub_2026, vars_laborales), lsuffix="_2025", rsuffix="_2026T1")
display(falt_sub)

Ocupados asalariados (antes de filtros de calidad): 2025 = 53,025 | 2026T1 = 13,258


,faltantes_2025,pct_faltantes_2025,faltantes_2026T1,pct_faltantes_2026T1
variable,,,,
edad,0,0.0,0,0.0
antiguedad_anios,0,0.0,0,0.0
antiguedad_meses,0,0.0,0,0.0
horas_semanales,0,0.0,0,0.0
nivel_educativo,0,0.0,0,0.0
dominio,0,0.0,0,0.0
salario_mensual,0,0.0,0,0.0


**¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una respuesta no registrada?**

- **La pregunta no corresponde (falta estructural).** El cuestionario tiene saltos: a una persona que no trabajó no se le pregunta la antigüedad, las horas ni el salario de su ocupación. Ese vacío significa "no aplica" y no es un error. Por eso, en la primera tabla, las variables laborales tienen porcentajes altos de faltantes: los archivos incluyen a todas las personas, ocupadas o no. Estos vacíos **no se imputan**; se resuelven definiendo bien la población analítica.
- **Respuesta no registrada (no respuesta o error de captura).** La persona sí estaba en el universo de la pregunta (ocupada y asalariada) pero el valor quedó vacío o inválido: no sabe, no quiso responder, error de captura. Esto sí es un problema de calidad y es lo que muestra la segunda tabla. Puede sesgar los resultados si quienes no declaran salario son distintos de quienes lo declaran.

Por esta razón los faltantes se miden sobre toda la base **y** sobre la población a la que aplica la pregunta. En el tratamiento de los datos, los registros que no permiten evaluar un criterio numérico se **excluyen y se contabilizan**, sin imputar el salario, y las categorías ausentes o no reconocidas se representan como `DESCONOCIDO`, sin convertirlas arbitrariamente a cero.

## 4. Unicidad de la clave `periodo_archivo`, `NUM_HOGAR`, `NUM_PERSONA`

Si aparecen claves repetidas dentro de un mismo período se investiga si son **repeticiones exactas** (todas las columnas seleccionadas idénticas) o **registros en conflicto** (mismas claves, valores distintos). No se usa `dropDuplicates()` para ocultar el problema.

In [15]:
CLAVE = ["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"]


def auditar_claves(df, etiqueta):
    cols_cmp = [c for c in df.columns if c != "archivo_origen"]
    n_total = df.count()
    n_claves = df.select(*CLAVE).distinct().count()
    n_nulos_clave = df.filter(F.col("NUM_HOGAR").isNull() | F.col("NUM_PERSONA").isNull()).count()

    grupos = (df.groupBy(*CLAVE)
              .agg(F.count("*").alias("n"),
                   F.size(F.collect_set(F.struct(*cols_cmp))).alias("n_versiones")))
    dup = grupos.filter("n > 1").cache()

    resumen = (dup.groupBy("periodo_archivo")
               .agg(F.count("*").alias("claves_duplicadas"),
                    F.sum("n").alias("filas_involucradas"),
                    F.sum(F.when(F.col("n_versiones") == 1, 1).otherwise(0)).alias("claves_repeticion_exacta"),
                    F.sum(F.when(F.col("n_versiones") > 1, 1).otherwise(0)).alias("claves_en_conflicto"))
               .orderBy("periodo_archivo").toPandas())

    print(f"[{etiqueta}] filas: {n_total:,} | claves distintas: {n_claves:,} | filas con NUM_HOGAR o NUM_PERSONA nulo: {n_nulos_clave:,}")
    if resumen.empty:
        print(f"[{etiqueta}] No hay claves duplicadas: la combinación es única.")
    else:
        display(resumen)
    return dup, resumen, cols_cmp


dup_raw, res_raw, cols_cmp = auditar_claves(df_todo, "antes de filtros, 2025 + 2026T1")

[antes de filtros, 2025 + 2026T1] filas: 253,519 | claves distintas: 253,519 | filas con NUM_HOGAR o NUM_PERSONA nulo: 0
[antes de filtros, 2025 + 2026T1] No hay claves duplicadas: la combinación es única.


In [16]:
# Si hay conflictos: ¿en qué variables difieren las filas con la misma clave? y ejemplos
if not res_raw.empty:
    conflictos = dup_raw.filter("n_versiones > 1").select(*CLAVE)
    if conflictos.count() > 0:
        filas_conf = df_todo.join(conflictos, on=CLAVE, how="inner")
        variables_cmp = [c for c in cols_cmp if c not in CLAVE]
        difiere = (filas_conf.groupBy(*CLAVE)
                   .agg(*[(F.size(F.collect_set(F.coalesce(F.col(c).cast("string"), F.lit("<NULO>")))) > 1).alias(c)
                          for c in variables_cmp]))
        conteo = difiere.agg(*[F.sum(F.col(c).cast("int")).alias(c) for c in variables_cmp]).toPandas().T
        conteo.columns = ["claves_donde_difiere"]
        print("Variables en las que difieren las filas en conflicto:")
        display(conteo[conteo["claves_donde_difiere"] > 0].sort_values("claves_donde_difiere", ascending=False))
        print("Ejemplos (hasta 10 filas):")
        display(filas_conf.orderBy(*CLAVE).limit(10).toPandas())
    exactas = dup_raw.filter("n_versiones = 1").select(*CLAVE)
    if exactas.count() > 0:
        print("Ejemplos de repeticiones exactas (hasta 6 filas):")
        display(df_todo.join(exactas, on=CLAVE, how="inner").orderBy(*CLAVE).limit(6).toPandas())

**Decisión sobre duplicados.** Los resultados anteriores se interpretan así:

- Si no hay claves repetidas, la combinación identifica un registro único dentro de cada período y no se elimina nada.
- Si hay repeticiones exactas, son la misma observación cargada más de una vez. Es el único caso que justificaría eliminarlas, pero solo después de documentarlo y decidirlo de forma explícita (no con `dropDuplicates()` sobre toda la tabla).
- Si hay registros en conflicto, no se puede saber cuál es el correcto. Se documentan las variables en las que difieren y se conservan tal como vienen, o se excluyen y contabilizan, aplicando siempre la misma regla.

Este notebook **no elimina** filas por duplicidad: cualquier decisión debe quedar reflejada arriba.

**¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del conjunto longitudinal?**

La ENEIC tiene un diseño longitudinal con rotación: parte de la muestra se vuelve a entrevistar en trimestres distintos. Una persona que aparece en dos períodos genera dos **observaciones distintas** (la misma persona en momentos distintos), cuyos valores pueden haber cambiado (empleo, horas, salario). Un duplicado real es la misma observación repetida **dentro de un mismo período**, que es lo que verifica la clave `periodo_archivo`, `NUM_HOGAR`, `NUM_PERSONA`.

Eliminar a una persona del segundo período descartaría información legítima, alteraría el tamaño y la composición de cada trimestre (dejarían de ser comparables) y podría sesgar los resultados. Dos consideraciones:

- Las observaciones repetidas de una misma persona **no son independientes**. Por eso train (2025) y test (2026T1) se separan por tiempo, y al validar dentro de 2025 conviene tener presente el riesgo de fuga de información con particiones aleatorias.
- Aquí solo se verifica la unicidad **dentro de cada período**; no se asume que `NUM_HOGAR` y `NUM_PERSONA` identifiquen a la misma persona entre períodos.

## 5. Población analítica: filtros secuenciales con conteo de exclusiones

Los filtros se aplican **siempre en el mismo orden**. En cada paso se cuenta cuántos registros se excluyen y, de ellos, cuántos se excluyen porque **no permiten evaluar** el criterio (dato nulo o no numérico). No se imputa el salario.

| Paso | Criterio |
|---|---|
| 1 | Edad finita y mayor o igual a 15 |
| 2 | Ocupado (`OCUPADOS = 1`; el diccionario solo define el valor 1, el vacío significa "no ocupado") |
| 3 | Asalariado (`P05C16` en 1, 2, 3 o 4) |
| 4 | Salario numérico, finito y estrictamente positivo (`P05D01 > 0`) |
| 5 | Meses de antigüedad: entero entre 0 y 11 |
| 6 | Años de antigüedad: finitos y no negativos |
| 7 | Antigüedad en años (`años + meses/12`) menor o igual a la edad |
| 8 | Horas habituales: mayores que 0 y menores o iguales a 168 por semana |

Como los pasos 1 a 3 delimitan primero a las personas a quienes sí corresponden las preguntas laborales, los vacíos que aparecen desde el paso 4 son respuestas **no registradas** y no faltas estructurales.

No se eliminan salarios extremos (altos o bajos) ni se aplican recortes por percentiles o transformaciones del objetivo. Su influencia se discutirá más adelante.

In [17]:
def finito(c):
    c = F.col(c) if isinstance(c, str) else c
    return c.isNotNull() & ~F.isnan(c) & (c != float("inf")) & (c != float("-inf"))


def definir_pasos():
    # Lista de (nombre, condición "evaluable", condición "cumple"), en orden fijo.
    # Paso 2: según el diccionario, OCUPADOS solo toma el valor 1; el vacío significa "no ocupado".
    # Por eso ese paso siempre es evaluable y los vacíos cuentan como excluidos, no como "no evaluables".
    edad, meses, anios = F.col("edad"), F.col("antiguedad_meses"), F.col("antiguedad_anios")
    horas, sal = F.col("horas_semanales"), F.col("salario_mensual")
    return [
        ("1. Edad finita y >= 15",           finito(edad),                    finito(edad) & (edad >= 15)),
        ("2. Ocupado (OCUPADOS = 1)",        F.lit(True),                     F.col("ocupado") == "1"),
        ("3. Asalariado (P05C16 en 1-4)",    F.col("categoria_ocupacional").isNotNull(),
                                             F.col("categoria_ocupacional").isin(list(CATEGORIAS_ASALARIADO))),
        ("4. Salario finito y > 0",          finito(sal),                     finito(sal) & (sal > 0)),
        ("5. Meses de antigüedad entero 0-11", finito(meses),
                                             finito(meses) & (meses == F.floor(meses)) & (meses >= 0) & (meses <= 11)),
        ("6. Años de antigüedad >= 0",       finito(anios),                   finito(anios) & (anios >= 0)),
        ("7. Antigüedad <= edad",            finito(anios) & finito(meses) & finito(edad),
                                             finito(anios) & finito(meses) & finito(edad) & ((anios + meses / 12) <= edad)),
        ("8. Horas habituales en (0, 168]",  finito(horas),                   finito(horas) & (horas > 0) & (horas <= 168)),
    ]


def aplicar_filtros(df):
    # Devuelve (DataFrame filtrado, tabla con antes/después/no evaluables por paso y período)
    registros, actual = [], df
    for nombre, evaluable, cumple in definir_pasos():
        ok = F.coalesce(cumple, F.lit(False))
        ev = F.coalesce(evaluable, F.lit(False))
        agg = (actual.groupBy("periodo_archivo")
               .agg(F.count("*").alias("antes"),
                    F.sum(ok.cast("int")).alias("despues"),
                    F.sum((~ev).cast("int")).alias("no_evaluables"))
               .toPandas())
        agg["paso"] = nombre
        registros.append(agg)
        actual = actual.filter(ok)
    return actual, pd.concat(registros, ignore_index=True)


filtrado_2025, tabla_2025 = aplicar_filtros(df_2025)
filtrado_2026, tabla_2026 = aplicar_filtros(df_2026)

In [18]:
def mostrar_pasos(tabla, titulo):
    orden = list(dict.fromkeys(tabla["paso"]))
    t = tabla.assign(excluidos=tabla["antes"] - tabla["despues"])
    exc = t.pivot(index="paso", columns="periodo_archivo", values="excluidos").reindex(orden)
    exc["TOTAL"] = exc.sum(axis=1)
    nev = t.pivot(index="paso", columns="periodo_archivo", values="no_evaluables").reindex(orden)
    nev["TOTAL"] = nev.sum(axis=1)
    rest = t.pivot(index="paso", columns="periodo_archivo", values="despues").reindex(orden)
    rest["TOTAL"] = rest.sum(axis=1)
    print(f"=== {titulo}: registros EXCLUIDOS en cada paso ===")
    display(exc)
    print(f"=== {titulo}: de los excluidos, cuántos NO PERMITEN EVALUAR el criterio (dato nulo o no numérico) ===")
    display(nev)
    print(f"=== {titulo}: registros que QUEDAN después de cada paso ===")
    display(rest)


mostrar_pasos(tabla_2025, "Train 2025")
mostrar_pasos(tabla_2026, "Test 2026T1")

=== Train 2025: registros EXCLUIDOS en cada paso ===


periodo_archivo,2025T1,2025T2,2025T3,2025T4,TOTAL
paso,,,,,
1. Edad finita y >= 15,16253,15882,15767,14984,62886
2. Ocupado (OCUPADOS = 1),13062,12942,13443,13121,52568
3. Asalariado (P05C16 en 1-4),8854,8851,8923,8569,35197
4. Salario finito y > 0,0,0,0,0,0
5. Meses de antigüedad entero 0-11,0,0,0,0,0
6. Años de antigüedad >= 0,0,0,0,0,0
7. Antigüedad <= edad,0,0,0,0,0
"8. Horas habituales en (0, 168]",0,0,0,0,0


=== Train 2025: de los excluidos, cuántos NO PERMITEN EVALUAR el criterio (dato nulo o no numérico) ===


periodo_archivo,2025T1,2025T2,2025T3,2025T4,TOTAL
paso,,,,,
1. Edad finita y >= 15,0,0,0,0,0
2. Ocupado (OCUPADOS = 1),0,0,0,0,0
3. Asalariado (P05C16 en 1-4),0,0,0,0,0
4. Salario finito y > 0,0,0,0,0,0
5. Meses de antigüedad entero 0-11,0,0,0,0,0
6. Años de antigüedad >= 0,0,0,0,0,0
7. Antigüedad <= edad,0,0,0,0,0
"8. Horas habituales en (0, 168]",0,0,0,0,0


=== Train 2025: registros que QUEDAN después de cada paso ===


...

periodo_archivo,2026T1,TOTAL
paso,,
1. Edad finita y >= 15,14803,14803
2. Ocupado (OCUPADOS = 1),13306,13306
3. Asalariado (P05C16 en 1-4),8476,8476
4. Salario finito y > 0,0,0
5. Meses de antigüedad entero 0-11,0,0
6. Años de antigüedad >= 0,0,0
7. Antigüedad <= edad,0,0
"8. Horas habituales en (0, 168]",0,0


=== Test 2026T1: de los excluidos, cuántos NO PERMITEN EVALUAR el criterio (dato nulo o no numérico) ===


periodo_archivo,2026T1,TOTAL
paso,,
1. Edad finita y >= 15,0,0
2. Ocupado (OCUPADOS = 1),0,0
3. Asalariado (P05C16 en 1-4),0,0
4. Salario finito y > 0,0,0
5. Meses de antigüedad entero 0-11,0,0
6. Años de antigüedad >= 0,0,0
7. Antigüedad <= edad,0,0
"8. Horas habituales en (0, 168]",0,0


=== Test 2026T1: registros que QUEDAN después de cada paso ===


periodo_archivo,2026T1,TOTAL
paso,,
1. Edad finita y >= 15,35040,35040
2. Ocupado (OCUPADOS = 1),21734,21734
3. Asalariado (P05C16 en 1-4),13258,13258
4. Salario finito y > 0,13258,13258
5. Meses de antigüedad entero 0-11,13258,13258
6. Años de antigüedad >= 0,13258,13258
7. Antigüedad <= edad,13258,13258
"8. Horas habituales en (0, 168]",13258,13258


In [19]:
def resumen_antes_despues(tabla):
    primero, ultimo = tabla["paso"].iloc[0], tabla["paso"].iloc[-1]
    a = tabla[tabla["paso"] == primero].set_index("periodo_archivo")["antes"]
    d = tabla[tabla["paso"] == ultimo].set_index("periodo_archivo")["despues"]
    r = pd.DataFrame({"registros_originales": a, "registros_finales": d})
    r["excluidos"] = r["registros_originales"] - r["registros_finales"]
    r["pct_retenido"] = (100 * r["registros_finales"] / r["registros_originales"]).round(2)
    return r


resumen_final = pd.concat([resumen_antes_despues(tabla_2025), resumen_antes_despues(tabla_2026)]).sort_index()
resumen_final.loc["TOTAL 2025"] = [
    resumen_final.loc[PERIODOS_TRAIN, "registros_originales"].sum(),
    resumen_final.loc[PERIODOS_TRAIN, "registros_finales"].sum(),
    resumen_final.loc[PERIODOS_TRAIN, "excluidos"].sum(),
    round(100 * resumen_final.loc[PERIODOS_TRAIN, "registros_finales"].sum()
          / resumen_final.loc[PERIODOS_TRAIN, "registros_originales"].sum(), 2)]
print("Registros por archivo antes y después de los filtros:")
display(resumen_final)

Registros por archivo antes y después de los filtros:


,registros_originales,registros_finales,excluidos,pct_retenido
periodo_archivo,,,,
2025T1,51588.0,13419.0,38169.0,26.01
2025T2,51167.0,13492.0,37675.0,26.37
2025T3,51583.0,13450.0,38133.0,26.07
2025T4,49338.0,12664.0,36674.0,25.67
2026T1,49843.0,13258.0,36585.0,26.60
TOTAL 2025,203676.0,53025.0,150651.0,26.03


**¿Por qué el número de registros de la base filtrada no representa a todos los trabajadores del país?**

- **Son registros de una muestra, no un censo.** La ENEIC es una encuesta con diseño muestral complejo. Cada registro representa a muchas personas, y esa equivalencia la da el factor de expansión `FACTOR`. Contar filas no equivale a contar trabajadores.
- **Los filtros seleccionan una subpoblación.** Quedan solo personas ocupadas, asalariadas (se excluyen patronos, cuenta propia y trabajadores no remunerados) y con un salario positivo registrado. Quienes no declaran salario o lo declaran en cero quedan fuera, y esa no respuesta probablemente no es aleatoria.
- **El análisis principal es no ponderado.** Cada registro pesa lo mismo, aunque unos representen a más personas que otros, por lo que las estadísticas describen los registros analizados y no son estimaciones oficiales de la población guatemalteca.
- **Hay observaciones repetidas.** Por la rotación de la muestra, una misma persona puede aparecer en más de un período, así que las filas acumuladas tampoco equivalen a personas distintas.

**Sobre `FACTOR`.** Se conserva en el conjunto preparado, pero no se utiliza en el clustering, los modelos ni las métricas principales. En un análisis poblacional se usaría como peso: los totales serían sumas de `FACTOR`, las medias serían medias ponderadas (`Σ FACTOR·x / Σ FACTOR`), y los percentiles y las proporciones también se ponderarían. Los errores estándar, además, requerirían la información del diseño (estratos y unidades primarias de muestreo), no solo el factor.

## 6. Variables categóricas: validación contra el diccionario y `DESCONOCIDO`

Antes de mapear, se observan los códigos que aparecen en los datos filtrados, para compararlos con el **diccionario de datos** de cada base. Los códigos válidos se tomaron del diccionario: `P03A03A` de 0 a 7 (0 = Ninguno, 1 = Preprimaria, 2 = Primaria, 3 = Básico, 4 = Diversificado, 5 = Superior, 6 = Maestría, 7 = Doctorado) y `DOMINIO` de 1 a 3 (Urbano Metropolitano, Resto Urbano, Rural Nacional). Un código presente en los datos pero ausente del diccionario se considera no reconocido. Los valores ausentes o no reconocidos se representan como `DESCONOCIDO`. El código educativo `0` significa "ninguno" y **no** es un faltante.

In [20]:
def tabla_codigos(df_a, df_b, var):
    # Conteo por código observado en cada período (los nulos se muestran como <nulo>)
    dd = df_a.unionByName(df_b)
    t = (dd.withColumn("codigo", F.coalesce(F.col(var), F.lit("<nulo>")))
         .groupBy("codigo", "periodo_archivo").count().toPandas())
    return (t.pivot(index="codigo", columns="periodo_archivo", values="count")
             .fillna(0).astype(int).sort_index())


for var in ["nivel_educativo", "dominio", "categoria_ocupacional"]:
    print(f"Códigos observados en {var} (población filtrada, antes de mapear):")
    display(tabla_codigos(filtrado_2025, filtrado_2026, var))

Códigos observados en nivel_educativo (población filtrada, antes de mapear):


periodo_archivo,2025T1,2025T2,2025T3,2025T4,2026T1
codigo,,,,,
0,1050,1044,958,943,1016
1,138,122,108,91,80
2,4062,4032,3950,3778,3957
3,2079,2080,2095,1995,2164
4,4196,4334,4332,3989,4117
5,1674,1663,1797,1684,1729
6,202,201,196,172,183
7,18,16,14,12,12


Códigos observados en dominio (población filtrada, antes de mapear):


periodo_archivo,2025T1,2025T2,2025T3,2025T4,2026T1
codigo,,,,,
1,5481,5581,5690,5320,5632
2,5167,5175,5076,4728,4846
3,2771,2736,2684,2616,2780


Códigos observados en categoria_ocupacional (población filtrada, antes de mapear):


periodo_archivo,2025T1,2025T2,2025T3,2025T4,2026T1
codigo,,,,,
1,1623,1634,1681,1637,1646
2,6825,7136,7566,7175,7503
3,4057,3702,3212,2871,3093
4,914,1020,991,981,1016


In [21]:
def resolver_codigos():
    # Conjunto de códigos válidos por variable. Si no se completó CODIGOS_VALIDOS, se usan
    # provisionalmente los códigos observados en 2025 (nunca los de 2026, para no filtrar información del test).
    validos = {"categoria_ocupacional": set(CATEGORIAS_ASALARIADO)}
    for var, lista in CODIGOS_VALIDOS.items():
        if lista is None:
            obs = sorted(r[0] for r in filtrado_2025.select(var).distinct().collect() if r[0] is not None)
            validos[var] = set(obs)
            print(f"AVISO: CODIGOS_VALIDOS['{var}'] no está definido. Se usan provisionalmente los códigos "
                  f"observados en 2025: {obs}. Compare con el diccionario y complete la configuración.")
        else:
            validos[var] = {normalizar_codigo(x) for x in lista}
    return validos


VALIDOS = resolver_codigos()


def preparar(df):
    # Construye la antigüedad en años y valida las variables categóricas
    df = df.withColumn("antiguedad", F.col("antiguedad_anios") + F.col("antiguedad_meses") / 12.0)
    for var, validos in VALIDOS.items():
        df = df.withColumn(
            var,
            F.when(F.col(var).isin(sorted(validos)), F.col(var)).otherwise(F.lit("DESCONOCIDO")))
    return df


COLUMNAS_FINALES = ["archivo_origen", "periodo_archivo", "anio_archivo", "trimestre_calendario",
                    "ANIO", "TRIMESTRE", "NUM_HOGAR", "NUM_PERSONA", "FACTOR",
                    "salario_mensual", "edad", "antiguedad", "horas_semanales",
                    "nivel_educativo", "categoria_ocupacional", "dominio",
                    "antiguedad_anios", "antiguedad_meses", "ocupado"]

train = preparar(filtrado_2025).select(*COLUMNAS_FINALES).cache()
test  = preparar(filtrado_2026).select(*COLUMNAS_FINALES).cache()
print(f"Conjunto preparado 2025 (train): {train.count():,} registros")
print(f"Conjunto preparado 2026T1 (test): {test.count():,} registros")

Conjunto preparado 2025 (train): 53,025 registros
Conjunto preparado 2026T1 (test): 13,258 registros


In [22]:
# Cuántos registros quedaron como DESCONOCIDO y verificación del código educativo 0
filas = []
for nombre, d in [("train 2025", train), ("test 2026T1", test)]:
    n = d.count()
    for var in ["nivel_educativo", "categoria_ocupacional", "dominio"]:
        k = d.filter(F.col(var) == "DESCONOCIDO").count()
        filas.append({"conjunto": nombre, "variable": var, "desconocido": k, "pct": round(100 * k / n, 3)})
display(pd.DataFrame(filas))

n0 = train.filter(F.col("nivel_educativo") == "0").count()
print(f"Registros de train con nivel_educativo = '0' (ninguno), conservados como categoría válida: {n0:,}")

,conjunto,variable,desconocido,pct
0,train 2025,nivel_educativo,0,0.0
1,train 2025,categoria_ocupacional,0,0.0
2,train 2025,dominio,0,0.0
3,test 2026T1,nivel_educativo,0,0.0
4,test 2026T1,categoria_ocupacional,0,0.0
5,test 2026T1,dominio,0,0.0


Registros de train con nivel_educativo = '0' (ninguno), conservados como categoría válida: 3,995


## 7. Verificación de unicidad en los conjuntos preparados

In [23]:
dup_prep, res_prep, _ = auditar_claves(train.unionByName(test), "conjuntos preparados, train + test")

[conjuntos preparados, train + test] filas: 66,283 | claves distintas: 66,283 | filas con NUM_HOGAR o NUM_PERSONA nulo: 0
[conjuntos preparados, train + test] No hay claves duplicadas: la combinación es única.


## 8. Guardado de los conjuntos preparados en Parquet

El conjunto de 2025 (train) y el de 2026T1 (test) se guardan **por separado**.

In [24]:
RUTA_TRAIN = DIR_SALIDA / "preparado" / "train_2025.parquet"
RUTA_TEST  = DIR_SALIDA / "preparado" / "test_2026T1.parquet"

train.write.mode("overwrite").parquet(str(RUTA_TRAIN))
test.write.mode("overwrite").parquet(str(RUTA_TEST))

# Verificación: se vuelve a leer desde disco
chk_train = spark.read.parquet(str(RUTA_TRAIN))
chk_test  = spark.read.parquet(str(RUTA_TEST))
print(f"train_2025.parquet : {chk_train.count():,} registros | períodos: {sorted(r[0] for r in chk_train.select('periodo_archivo').distinct().collect())}")
print(f"test_2026T1.parquet: {chk_test.count():,} registros | períodos: {sorted(r[0] for r in chk_test.select('periodo_archivo').distinct().collect())}")
assert chk_train.count() == train.count() and chk_test.count() == test.count()
assert set(r[0] for r in chk_test.select("periodo_archivo").distinct().collect()) == {PERIODO_TEST}
chk_train.printSchema()

train_2025.parquet : 53,025 registros | períodos: ['2025T1', '2025T2', '2025T3', '2025T4']
test_2026T1.parquet: 13,258 registros | períodos: ['2026T1']
root
 |-- archivo_origen: string (nullable = true)
 |-- periodo_archivo: string (nullable = true)
 |-- anio_archivo: integer (nullable = true)
 |-- trimestre_calendario: integer (nullable = true)
 |-- ANIO: integer (nullable = true)
 |-- TRIMESTRE: integer (nullable = true)
 |-- NUM_HOGAR: string (nullable = true)
 |-- NUM_PERSONA: string (nullable = true)
 |-- FACTOR: double (nullable = true)
 |-- salario_mensual: double (nullable = true)
 |-- edad: double (nullable = true)
 |-- antiguedad: double (nullable = true)
 |-- horas_semanales: double (nullable = true)
 |-- nivel_educativo: string (nullable = true)
 |-- categoria_ocupacional: string (nullable = true)
 |-- dominio: string (nullable = true)
 |-- antiguedad_anios: double (nullable = true)
 |-- antiguedad_meses: double (nullable = true)
 |-- ocupado: string (nullable = true)



## Resumen de la Parte 1

- Los cinco archivos se leyeron por separado, se homologaron por nombre de columna y por tipo, y se guardaron en Parquet con `archivo_origen`, `periodo_archivo`, `anio_archivo` y `trimestre_calendario`.
- Los cuatro archivos de 2025 se unieron con `unionByName`; 2026T1 se mantuvo separado.
- Los filtros se aplicaron siempre en el mismo orden, con conteo de exclusiones y de registros no evaluables por paso.
- La unicidad de la clave `periodo_archivo`, `NUM_HOGAR`, `NUM_PERSONA` se auditó sin usar `dropDuplicates()`.
- Los conjuntos preparados quedaron en `salidas/preparado/train_2025.parquet` y `salidas/preparado/test_2026T1.parquet`, con `antiguedad` en años, categóricas validadas (`DESCONOCIDO` cuando corresponde) y `FACTOR` conservado sin usarse.

Las partes siguientes (estadística descriptiva, correlaciones y clustering) parten de estos dos archivos Parquet.